### OCI Data Science - Useful Tips
<details>
<summary><font size="2">Check for Public Internet Access</font></summary>

```python
import requests
response = requests.get("https://oracle.com")
assert response.status_code==200, "Internet connection failed"
```
</details>
<details>
<summary><font size="2">Helpful Documentation </font></summary>
<ul><li><a href="https://docs.cloud.oracle.com/en-us/iaas/data-science/using/data-science.htm">Data Science Service Documentation</a></li>
<li><a href="https://docs.cloud.oracle.com/iaas/tools/ads-sdk/latest/index.html">ADS documentation</a></li>
</ul>
</details>
<details>
<summary><font size="2">Typical Cell Imports and Settings for ADS</font></summary>

```python
%load_ext autoreload
%autoreload 2
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(format='%(levelname)s:%(message)s', level=logging.ERROR)

import ads
from ads.dataset.factory import DatasetFactory
from ads.automl.provider import OracleAutoMLProvider
from ads.automl.driver import AutoML
from ads.evaluations.evaluator import ADSEvaluator
from ads.common.data import ADSData
from ads.explanations.explainer import ADSExplainer
from ads.explanations.mlx_global_explainer import MLXGlobalExplainer
from ads.explanations.mlx_local_explainer import MLXLocalExplainer
from ads.catalog.model import ModelCatalog
from ads.common.model_artifact import ModelArtifact
```
</details>
<details>
<summary><font size="2">Useful Environment Variables</font></summary>

```python
import os
print(os.environ["NB_SESSION_COMPARTMENT_OCID"])
print(os.environ["PROJECT_OCID"])
print(os.environ["USER_OCID"])
print(os.environ["TENANCY_OCID"])
print(os.environ["NB_REGION"])
```
</details>

## Image search in OCI OpenSearch using OpenAI CLIP (Contrasive Language-Image Pretraining) model.

This notebook shows how you can integrate OpenAI CLIP model from Hugging Face into OCI OpenSearch for image search functionality. Open AI CLIP model is a multi modal vision and language model. In this example we are going to use CLIP model to generate embeddings for each image and then these embeddings will be stored on OCI OpenSearch vector store. When the user perform searches using text, we will use the CLIP model again to generate the embeddings and then perform a comparison with OpenSearch vector index. OCI OpenSearch will return the matches which will contain the image path and similarity score.

### Prerequisites
1. OCI OpenSearch cluster running inside a VCN. If you do not have OpenSearch cluster then follow the OCI documentation [Creating an OpenSearch Cluster](https://docs.oracle.com/en-us/iaas/Content/search-opensearch/Tasks/creatingsearchclusters.htm).
2. OCI Data science tenancy to run the jupyter notebook. If you do not have Data science tenancy setup then follow the OCI documentation [Manually Configuring a Data Science Tenancy](https://docs.oracle.com/en-us/iaas/data-science/data-science-tutorial/get-started.htm).

You can find the details of CLIP model [here](https://huggingface.co/docs/transformers/model_doc/clip).

### Install all dependencies.

pip install torch transformers opensearch-py

### Import required libraries and set global variables.

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests.auth import HTTPBasicAuth
import matplotlib.pyplot as plt

### General settings and OpenSearch settings (Modify these according to your environment).

- IMAGES_FOLDER : Folder path where the images are stored. Currently '.jpg', '.png', '.jpeg' are supported.
- MODEL_NAME : CLIP model id to download the model from Hugging Face

In [ ]:
# General settings
IMAGES_FOLDER = "samples/"
MODEL_NAME = "openai/clip-vit-base-patch32"

# OCI OpenSearch settings
OPENSEARCH_PORT = 9200
OPENSEARCH_HOST = "xxxx.opensearch.us-ashburn-1.oci.oraclecloud.com"
OPENSEARCH_USER = "xxccc"
OPENSEARCH_PASS = "xxxccc"
OPENSEARCH_INDEX = "image_search"

### Connect to OCI OpenSearch and load pretrained CLIP model and processor from Hugging Face.

In [ ]:
# Connect to OCI OpenSearch
client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_auth=HTTPBasicAuth(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection
)

# Load the pretrained CLIP model and processor
model = CLIPModel.from_pretrained(MODEL_NAME)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

### Function to generate image embeddings.

In [ ]:
def encode_image(image_path):
    # Open image
    image = Image.open(image_path)
    # Process the image
    inputs = processor(images=image, return_tensors="pt")
    # Get image embeddings
    with torch.no_grad():  # Disable gradient calculation for inference
        image_embeddings = model.get_image_features(**inputs)

    return image_embeddings.squeeze().tolist()

### Function to generate text embeddings.

In [ ]:
def encode_text(text):
    inputs = processor(text=[text], return_tensors="pt")
    with torch.no_grad():
        text_features = model.get_text_features(**inputs)
    return text_features.squeeze().tolist()

### Function to store image path and embeddings to OCI OpenSearch cluster.

In [ ]:
# Function to store image embeddings in OpenSearch
def store_image_embeddings(image_folder):
    print("Ingesting image data ....")
    image_paths = [os.path.join(image_folder, fname) for fname in os.listdir(image_folder) if fname.endswith(('.jpg', '.png', '.jpeg'))]

    for image_path in image_paths:
        embedding = encode_image(image_path)
        document = {
            "image_path": image_path,
            "embedding": embedding  # Store as embedding
        }
        client.index(index=INDEX_NAME, body=document)

    print(f"Stored {len(image_paths)} images in OCI OpenSearch.")

### Function to perform a vector search in OCI OpenSearch.

In [ ]:
# Function to search for the most similar image using text
def search_images_by_text(query_text, top_k=1):
    query_embedding = encode_text(query_text)
    #print(query_embedding)

    # OpenSearch KNN Search Query
    query = {
        "size": top_k,
        "query": {
            "knn": {
                "embedding": {
                    "vector": query_embedding,
                    "k": top_k
                }
            }
        }
    }
   
    #print(query)
    response = client.search(index=OPENSEARCH_INDEX, body=query)
    results = [(hit["_source"]["image_path"], hit["_score"]) for hit in response["hits"]["hits"]]
    return results

### Function to create index with proper mappings in OCI OpenSearch.

In [ ]:
# Function to create OpenSearch index with correct mapping
def create_index():
    index_mapping = {
      "settings": {
        "index": {
          "knn": "true",
          "knn.algo_param.ef_search": 100,
          "number_of_shards": 1,
          "number_of_replicas": 0
        }
      },        
      "mappings": {
        "properties": {
          "image_path": {
            "type": "text"
          },
          "embedding": {
            "type": "knn_vector",
            "dimension": 512
          }
        }
      }
    }
    
    if client.indices.exists(OPENSEARCH_INDEX):
        print(f"Index '{OPENSEARCH_INDEX}' already exists. Deleting and recreating...")
        client.indices.delete(index=OPENSEARCH_INDEX)

    response = client.indices.create(index=OPENSEARCH_INDEX, body=index_mapping)
    print(f"Index '{OPENSEARCH_INDEX}' created successfully:", response)    

### Functions to show results based on the results returned from OCI OpenSearch.

In [ ]:
# Function to show the result image
def show_image(image_path):
    # Load the image
    image = Image.open(image_path)

    # Resize the image to 256x256
    resized_image = image.resize((256, 256))
   
    # Display the resized image with correct dimensions
    plt.figure(figsize=(2, 2))  # Adjust figure size (2x2 inches to match 128x128)
    plt.imshow(resized_image)
    plt.axis("off")  # Hide axes
    plt.show()
    
def show_results(results):
    for img_path, score in results:
        normalized_score = score * 100
        if normalized_score > 0.6: # For testing only
            print(f"Matched Image path: {img_path} \nSimilarity Score: {normalized_score:.2f}%")
            show_image(img_path)
        else:
            print("No match")

### Create index and store embeddings in OpenSearch.

In [ ]:
# Create index with mappings
create_index()

# Get all images, generate embeddings and ingest into OpenSearch
store_image_embeddings(IMAGES_FOLDER)

### Perform search using text and show result image and similarity score.

Here are some scenarios to test.
- Similarity score changes according to the search text (You can see the similarity score increases as we add more details to search text. For example, the search text 'a person wearing a hoodie' got a similarity score of 0.72 and the search text 'a person wearing a black hoodie and a blue cap' got greater similarity score of 0.83.).
- The higher the similarity score, the better the text matches the image.
- Spelling mistake in search query (Still you get results back with low similarity score).
- General search for objects within the image (For example. 'a room with a fan').

In [ ]:
#query_text = "a room with a fan"
#query_text = "a person wearing a hoodie"
#query_text = "a person wearing a black hoodie"
#query_text = "a person with a cap"
#query_text = "a person wtih a cap" # wrong spelling wtih
#query_text = "a person with blue cap"
#query_text = "a person wearing a black hoodie and a blue cap"
#query_text = "a keyboard"
query_text = "a mechanical keyboard"
#query_text = "a computer keyboard"
#query_text = "a remote control"
#query_text = "a black remote control"
#query_text = "a black remote control with rounded buttons"


results = search_images_by_text(query_text)
show_results(results)